# Station Matching


## Environment set-up

In [1]:
from shapely.geometry import Point
from shapely.ops import nearest_points

from functools import reduce
import datetime
from pandas import *
import boto3
import geopandas as gpd
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from io import BytesIO, StringIO

## New logger function
from log_config import logger

# Import qaqc stage calc functions
try:
    from QAQC_pipeline import *
except:
    print("Error importing QAQC_pipeline.py")

# import tempfile  # Used for downloading (and then deleting) netcdfs to local drive from s3 bucket
import os

# Silence warnings
import warnings
from shapely.errors import ShapelyDeprecationWarning

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings(
    "ignore", category=ShapelyDeprecationWarning
)  # Warning is raised when creating Point object from coords. Can't figure out why.

plt.rcParams["figure.dpi"] = 300

In [2]:
# AWS credentials
s3 = boto3.resource("s3")
s3_cl = boto3.client("s3")

## AWS buckets
bucket = "wecc-historical-wx"
qaqcdir = "3_qaqc_wx/"
mergedir = "4_merge_wx/"

## Step 1: Identify candidates for concatenation and upload to AWS

We do so by identifying stations with exactly matching latitudes and longitudes.

In [3]:
# A list of networks to be checked for concatenation
target_networks = ["ASOSAWOS","VALLEYWATER", "MARITIME"]

In [4]:
def concatenation_check(station_list):
    """
    This function flags stations that need to be concatenated.

    Rules
    ------
        1.) Stations are flagged if they have identical latitudes and longitudes

    Parameters
    ------
        station_list: pd.DataFrame
            list of station information

    Returns
    -------
        if success:
            new_station_list: pd.DataFrame
                input station list with a flag column assigning an integer to each group of repeat latitudes and longitudes

        if failure:
            None

    """
    ##### Flag stations with identical latitudes and longitudes, then assign each group a unique integer

    # List of possible variable names for longitudes and latitudes
    lat_lon_list = ["LAT", "LON", "latitude", "longitude", "LATITUDE", "LONGITUDE", 'lat','lon']
    # Extract the latitude and longitude variable names from the input dataframe
    lat_lon_cols = [col for col in station_list.columns if col in lat_lon_list]

    # Generate column flagging duplicate latitudes and longitudes
    station_list["concat_subset"] = station_list.duplicated(
        subset=lat_lon_cols, keep=False
    )
    # within each group of identical latitudes and longitudes, assign a unique integer
    station_list["concat_subset"] = (
        station_list[station_list["concat_subset"] == True].groupby(lat_lon_cols).ngroup()
    )

    ##### Order station list by flag
    concat_station_list = station_list.sort_values("concat_subset")

    ##### Keep only flagged stations
    concat_station_list = concat_station_list[~concat_station_list["concat_subset"].isna()]

    ##### Format final list
    # Convert flags to integers - this is necessary for the final concatenation step
    concat_station_list["concat_subset"] = concat_station_list["concat_subset"].astype(
        "int32"
    )
    # Now keep only the ERA-ID and flag column
    era_id_list = ['ERA-ID','era-id']
    era_id_col = [col for col in station_list.columns if col in era_id_list]
    concat_station_list = concat_station_list[era_id_col + ["concat_subset"]]

    # Standardize ERA id to "ERA-ID" (this is specific to Valleywater stations)
    if 'era-id' in era_id_col:
        concat_station_list.rename(columns={"era-id": "ERA-ID"}, inplace=True)

    return concat_station_list

In [5]:
def apply_concat_check(station_names_list):
    """
    This function applies the conatenation check to a list of target stations. 
    It then upload a csv containing the ERA IDs and concatenation subset ID for 
    all identified stations in a network.

    Parameters
    ------
        station__names_list: pd.DataFrame
            list of target station names

    Returns
    -------
        if success:
            uploads list of stations to be concatenated to AWS
        if failure:
            None

    """
    final_list = pd.DataFrame([])
    for station in station_names_list:

        ##### Import station list of target station
        key = "2_clean_wx/{}/stationlist_{}_cleaned.csv".format(station,station)
        bucket_name = "wecc-historical-wx"
        list_import = s3_cl.get_object(
            Bucket=bucket,
            Key=key,
        )
        station_list = pd.read_csv(BytesIO(list_import["Body"].read()))

        ##### Apply concatenation check
        concat_list = concatenation_check(station_list)

        ##### Rename the flags for each subset to <station>_<subset number>
        concat_list["concat_subset"] = station + '_' + concat_list["concat_subset"].astype(str)

        ##### Append to final list of stations to concatenate
        final_list = pd.concat([final_list,concat_list])

        ##### Upload to QAQC directory in AWS
        new_buffer = StringIO()
        final_list.to_csv(new_buffer, index = False)
        content = new_buffer.getvalue()

        # the csv is stored in each station folder within 3_qaqc_wx
        s3_cl.put_object(
            Bucket = bucket_name,
            Body = content,
            Key = qaqcdir + station + "/concat_list_{}.csv".format(station)
        )
        
    return None

In [6]:
apply_concat_check(target_networks)

## Step 2: Concatenate Stations

In [7]:
# idea
# identify unique station-"pairs"
# for each station pair
# retrieve the set of stations associated with that pair
# if == 2, do the following
# if == 3, ....
# if == 6, ....

In [276]:
url_1 = "s3://wecc-historical-wx/3_qaqc_wx/{}/{}.zarr".format("MARITIME", "MARITIME_SMOC1")
url_2 = "s3://wecc-historical-wx/3_qaqc_wx/{}/{}.zarr".format("MARITIME", "MARITIME_ICAC1")

print('Retrieving....', url_1)
print('Retrieving....', url_2)
ds_1 = xr.open_zarr(url_1)
ds_2 = xr.open_zarr(url_2)

df_1, MultiIndex_1, attrs_1, var_attrs_1, era_qc_vars_1 = qaqc_ds_to_df(ds_1, verbose=False)
df_2, MultiIndex_2, attrs_2, var_attrs_2, era_qc_vars_2 = qaqc_ds_to_df(ds_2, verbose=False)

Retrieving.... s3://wecc-historical-wx/3_qaqc_wx/MARITIME/MARITIME_SMOC1.zarr
Retrieving.... s3://wecc-historical-wx/3_qaqc_wx/MARITIME/MARITIME_ICAC1.zarr


In [280]:
df_3 = pd.merge(df_1, df_2, how='outer')
df_3

,time,anemometer_height_m,elevation,elevation_eraqc,lat,lon,ps,ps_eraqc,sfcWind,sfcWind_dir,...,sfcWind_eraqc,tas,tas_eraqc,thermometer_height_m,station,hour,day,month,year,date
0,2005-04-01 02:00:00,NaN,0.0,NaN,34.008,-118.5,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,MARITIME_SMOC1,2,1,4,2005,2005-04-01
1,2005-04-01 03:00:00,NaN,0.0,NaN,34.008,-118.5,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,MARITIME_SMOC1,3,1,4,2005,2005-04-01
2,2005-04-01 04:00:00,NaN,0.0,NaN,34.008,-118.5,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,MARITIME_SMOC1,4,1,4,2005,2005-04-01
3,2005-04-01 05:00:00,NaN,0.0,NaN,34.008,-118.5,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,MARITIME_SMOC1,5,1,4,2005,2005-04-01
4,2005-04-01 06:00:00,NaN,0.0,NaN,34.008,-118.5,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,MARITIME_SMOC1,6,1,4,2005,2005-04-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1367105,2022-08-31 23:30:00,13.6,6.6,NaN,34.008,-118.5,100850.0,NaN,NaN,NaN,...,NaN,295.55,NaN,13.2,MARITIME_ICAC1,23,31,8,2022,2022-08-31
1367106,2022-08-31 23:36:00,13.6,6.6,NaN,34.008,-118.5,100840.0,NaN,NaN,NaN,...,NaN,295.35,NaN,13.2,MARITIME_ICAC1,23,31,8,2022,2022-08-31
1367107,2022-08-31 23:42:00,13.6,6.6,NaN,34.008,-118.5,100810.0,NaN,NaN,NaN,...,NaN,295.65,NaN,13.2,MARITIME_ICAC1,23,31,8,2022,2022-08-31
1367108,2022-08-31 23:48:00,13.6,6.6,NaN,34.008,-118.5,100790.0,NaN,NaN,NaN,...,NaN,296.25,NaN,13.2,MARITIME_ICAC1,23,31,8,2022,2022-08-31


In [282]:
df_3['station'].unique()[0]

'MARITIME_SMOC1'

In [271]:
def _multiindex_concat_nooverlap(m_old, m_new, name):
    """
    annoying handling but must be done
    ORDER IS IMPORTANT

    docstring needed, feeling lazy
    """

    # combine time indices of two multiindexes
    tidx = pd.concat([pd.Series(m_old.get_level_values('time').values), pd.Series(m_new.get_level_values('time').values)]).reset_index().drop(columns='index')

    # idenitify if there are duplicate times
    tidx = tidx.rename(columns = {0: 'time'})
    tidx = tidx.sort_values('time').drop_duplicates(subset=["time"])

    # PULL the station name from m_new and set to the same length
    stnidx = pd.Series(name, index = np.arange(len(tidx_concat)), name='station').reset_index().drop(columns='index')

    # combine into new df (ugh)
    df_ugh = pd.concat([stnidx, tidx], axis=1)
    
    return df_ugh

In [341]:
def _ds_order_check(ds_1, ds_2):
    """
    docstring needed, feeling lazy
    """

    # convert to dataframes with corresponding information
    df_1, MultiIndex_1, attrs_1, var_attrs_1, era_qc_vars_1 = qaqc_ds_to_df(ds_1, verbose=False)
    df_2, MultiIndex_2, attrs_2, var_attrs_2, era_qc_vars_2 = qaqc_ds_to_df(ds_2, verbose=False)
    
    # determine which dataset is older
    if df_1["time"].max() < df_2["time"].max():
        # if df_1 has an earlier end tiem than df_2, then d_2 is newer
        # we also grab the name of the newer station in this step, for use later
        df_new = df_2
        attrs_new = attrs_2
        df_old = df_1

    else:
        df_new = df_1
        attrs_new = attrs_1
        df_old = df_2

    stn_n_to_keep = df_new['station'].unique()[0]
    stn_n_to_drop = df_old['station'].unique()[0]
    print(f'Station will be concatenated and saved as: {stn_n_to_keep}')
    
    # now set things up to determine if there is temporal overlap between df_new and df_old
    df_overlap = df_new[df_new["time"].isin(df_old["time"])]

    # if there is no overlap between the two time series, just concatenate
    if len(df_overlap) == 0:
        print('No overlap!')
        df_concat = pd.merge(df_old, df_new, how='outer')
        df_concat['station'] = stn_n_to_keep

        
    # if overlap exists, split into subsets and concatenate
    else:
        print('crap, there is overlap')
        # ##### Split datframes into subsets #####

        # # Remove data in time overlap between old and new
        # df_old_cleaned = df_old[~df_old["time"].isin(df_overlap["time"])]
        # df_new_cleaned = df_new[~df_new["time"].isin(df_overlap["time"])]

        # ##### Concatenate subsets #####
        # df_concat = pd.concat([df_old_cleaned, df_overlap, df_new_cleaned])
        # print(len(df_concat))

    return df_concat, stn_n_to_keep, stn_n_to_drop, attrs_new


def _concat_export_help(ds_concat, network_name, attrs_new, station_name_new, station_name_old):
    """
    docstring needed, feeling lazy
    """
    
    #### Prepare for export #####
    # Convert datatype of station coordinate
    ds_concat.coords["station"] = ds_concat.coords["station"].astype("<U20")

    ## Include past attributes -- do this manually?
    for i in attrs_new:
        ds_concat.attrs[i] = attrs_new[i]

    # Update 'history' attribute
    timestamp = datetime.datetime.utcnow().strftime("%m-%d-%Y, %H:%M:%S")
    ds_concat.attrs["history"] = ds_concat.attrs[
        "history"
    ] + " \nstation_matching.ipynb run on {} UTC".format(timestamp)

    # Update 'comment' attribute
    ds_concat.attrs["comment"] = (
        "Intermediary data product. This data has been subjected to cleaning, QA/QC, but may not have been standardized."
    )

    # Add new qaqc_files_merged attribute
    ds_concat.attrs["qaqc_files_merged"] = (
        "{}, {} merged. Overlap retained from newer station data.".format(
            station_name_old, station_name_new
        )
    )

    ## Export ###
    # ! a test name is used below
    # ! the final name will be that of the newer dataframe
    export_url = "s3://wecc-historical-wx/3_qaqc_wx/{}/{}_{}.zarr".format(
        network_name, "test_concat", station_name_new
    )
    print(export_url)
    # ds_concat.to_zarr(export_url, mode="w") ## WHEN READY TO EXPORT
    
    return None


def concatenate_station_pairs2(network_name):
    """
    docstring needed, feeling lazy
    """
    
    # read in full concat station list
    print(network_name)
    concat_list = pd.read_csv(f"s3://wecc-historical-wx/3_qaqc_wx/{network_name}/concat_list_{network_name}.csv")

    # identify stns within designated network
    concat_by_network = concat_list.loc[concat_list.concat_subset.str.contains(network_name)]
    unique_pair_names = concat_by_network.concat_subset.unique()
    print(f'There are {len(concat_by_network)} stations to be concatenated into {len(unique_pair_names)} station pairs within {network_name}...')

    #### removing the first two becuase they're actually separate stations
    unique_pair_names = unique_pair_names[1:]

    # set up pairs
    for pair in unique_pair_names:
        print(pair)
        # pull out stations corresponding to pair name
        stns_to_pair = concat_by_network.loc[concat_by_network.concat_subset == pair]

        if len(stns_to_pair) == 2: # 2 stations to concat together
            print('\n', stns_to_pair)

            # import this subset of datasets and convert to dataframe
            url_1 = "s3://wecc-historical-wx/3_qaqc_wx/{}/{}.zarr".format(network_name, stns_to_pair.iloc[0]['ERA-ID'])
            url_2 = "s3://wecc-historical-wx/3_qaqc_wx/{}/{}.zarr".format(network_name, stns_to_pair.iloc[1]['ERA-ID'])

            print('Retrieving....', url_1)
            print('Retrieving....', url_2)
            ds_1 = xr.open_zarr(url_1)
            ds_2 = xr.open_zarr(url_2)

            # identify order
            df_concat, stn_n_to_keep, stn_n_to_drop, attrs_new = _ds_order_check(ds_1, ds_2)

            # now start concat 
            df_concat = df_concat.drop(["hour", "day", "month", "year", "date"], axis=1)
            df_to_export = df_concat.set_index(['station', 'time'])
            
            ## Convert concatenated dataframe to dataset -- seeing duplicate timestamps here -- the exact same length as df2?
            ds_to_export = df_to_export.to_xarray()
            _concat_export_help(ds_to_export, network_name, attrs_new, stn_n_to_keep, stn_n_to_drop)

        else: 
            print('whoa... that is a lot of stations')

    return ds_to_export

In [342]:
ds = concatenate_station_pairs2('MARITIME')
ds

MARITIME
There are 7 stations to be concatenated into 3 station pairs within MARITIME...
MARITIME_1

             ERA-ID concat_subset
54  MARITIME_ICAC1    MARITIME_1
55  MARITIME_SMOC1    MARITIME_1
Retrieving.... s3://wecc-historical-wx/3_qaqc_wx/MARITIME/MARITIME_ICAC1.zarr
Retrieving.... s3://wecc-historical-wx/3_qaqc_wx/MARITIME/MARITIME_SMOC1.zarr
df1 1084742
df2 282368
1084742 282368 1367110
Station will be concatenated and saved as: MARITIME_ICAC1
No overlap!
s3://wecc-historical-wx/3_qaqc_wx/MARITIME/test_concat_MARITIME_ICAC1.zarr
MARITIME_2
whoa... that is a lot of stations


<xarray.Dataset>
Dimensions:               (station: 1, time: 1367110)
Coordinates:
  * station               (station) <U20 'MARITIME_ICAC1'
  * time                  (time) datetime64[ns] 2005-04-01T02:00:00 ... 2022-...
Data variables: (12/14)
    anemometer_height_m   (station, time) float64 nan nan nan ... 13.6 13.6 13.6
    elevation             (station, time) float64 0.0 0.0 0.0 ... 6.6 6.6 6.6
    elevation_eraqc       (station, time) float64 nan nan nan ... nan nan nan
    lat                   (station, time) float64 34.01 34.01 ... 34.01 34.01
    lon                   (station, time) float64 -118.5 -118.5 ... -118.5
    ps                    (station, time) float64 nan nan ... 1.008e+05
    ...                    ...
    sfcWind_dir           (station, time) float64 nan nan nan ... nan nan nan
    sfcWind_dir_eraqc     (station, time) float64 nan nan nan ... nan nan nan
    sfcWind_eraqc         (station, time) float64 nan nan nan ... nan nan nan
    tas                   (station, time) float64 nan nan nan ... 296.2 296.6
    tas_eraqc             (station, time) float64 nan nan nan ... nan nan nan
    thermometer_height_m  (station, time) float64 nan nan nan ... 13.2 13.2 13.2
Attributes: (12/14)
    anemometer_height_m:    13.6
    barometer_elevation_m:  8.2
    citation:               
    comment:                Intermediary data product. This data has been sub...
    disclaimer:             This document was prepared as a result of work sp...
    history:                MARITIME_clean.py script run on 05-19-2023, 18:27...
    ...                     ...
    raw_files_merged:       23
    source:                 
    station_name:           9410840 - SANTA MONICA PIER
    thermometer_height_m:   13.2
    title:                  MARITIME quality controlled
    qaqc_files_merged:      MARITIME_SMOC1, MARITIME_ICAC1 merged. Overlap re...

----
VANESSA'S CODE BELOW

### The functions

In [ ]:
def concatenate_station_pairs(network_name):
    """
    Concatenates two input datasets, deletes the originals, and exports the final concatenated dataset. 
    Also returns a list of the ERA-IDs of all stations that are concatenated.

    Rules
    ------
        1.) concatenation: keep the newer station data in the time range in which both stations overlap

    Parameters
    ------
        network_name: string
            weather station network

    Returns
    -------
        if success: 
            return list of ERA-IDs are stations that are concatenated
            all processed datasets are exported to the merge folder in AWS and the original datasets are deleted
        if failure:
            None
    """
    
    ##### Read in concatenation list of input network
    print(network_name)
    concat_list = pd.read_csv(f"s3://wecc-historical-wx/3_qaqc_wx/{network_name}/concat_list_{network_name}.csv")

    # ! you can truncate the concat list here, for testing
    concat_list = concat_list.tail(2)
    print(concat_list)
    # ! end

    subset_number = len(concat_list['concat_subset'].unique())
    print('subset_number', subset_number) # length of df with unique stn PAIRS in concat_list (or it's really subset_number + 1 to indicate # of stns that require subsetting)

    # isolate only the networks id'd by input network network_name
    

    # initiate empty list, to which we will iteratively add the ERA-IDs of stations that are concatenated
    final_concat_list = []

    for i in range(0,3):
        print('starting i', i)

        subset_i = concat_list.loc[concat_list["ERA-ID"].isin([network_name])]
        print(len(subset_i))

        # # count the number of stations in subset i
        # subset_i = concat_list[
        #     concat_list["concat_subset"].str.contains("{}".format(i))
        # ] ## isn't this the same thing as subset_number + 1 ??
        # print('Subset_i', subset_i) # this is empty

        # n = subset_i.count()[0]
        # print('n', n)

        ## resetting for testing
        n = 2

        # if there are only two stations, proceed with concatenation
        if n == 2:
            try: 
                # retrieve ERA IDs in this subset of stations
                station_1 = concat_list.iloc[0]["ERA-ID"].item()
                station_2 = concat_list.iloc[1]["ERA-ID"].item()
                print(station_1, station_2)

                # import this subset of datasets and convert to dataframe
                url_1 = "s3://wecc-historical-wx/3_qaqc_wx/{}/{}.zarr".format(
                    network_name, station_1
                )
                url_2 = "s3://wecc-historical-wx/3_qaqc_wx/{}/{}.zarr".format(
                    network_name, station_2
                )

                ds_1 = xr.open_zarr(url_1)
                ds_2 = xr.open_zarr(url_2)

                df_1, MultiIndex_1, attrs_1, var_attrs_1, era_qc_vars_1 = qaqc_ds_to_df(ds_1, verbose=False)
                df_2, MultiIndex_2, attrs_2, var_attrs_2, era_qc_vars_2 = qaqc_ds_to_df(ds_2, verbose=False)

                # determine which dataset is older
                if df_1["time"].max() < df_2["time"].max():
                    # if df_1 has an earlier end tiem than df_2, then d_2 is newer
                    # we also grab the name of the newer station in this step, for use later
                    df_new = df_2
                    ds_new = ds_2
                    MultiIndex_new = MultiIndex_2
                    attrs_new = attrs_2

                    df_old = df_1
                    ds_old = ds_1
                    MultiIndex_old = MultiIndex_1

                else:
                    df_new = df_1
                    ds_new = df_1
                    MultiIndex_new = MultiIndex_2
                    attrs_new = attrs_2

                    df_old = df_2
                    ds_old = ds_2
                    MultiIndex_old = MultiIndex_2

                # now set things up to determine if there is temporal overlap between df_new and df_old
                df_overlap = df_new[df_new["time"].isin(df_old["time"])]

                # if there is no overlap between the two time series, just concatenate
                if len(df_overlap) == 0:
                    df_concat = concat([df_old, df_new])

                # if not, split into subsets and concatenate
                else:
                    ##### Split datframes into subsets #####

                    # Remove data in time overlap between old and new
                    df_old_cleaned = df_old[~df_old["time"].isin(df_overlap["time"])]
                    df_new_cleaned = df_new[~df_new["time"].isin(df_overlap["time"])]

                    ##### Concatenate subsets #####
                    df_concat = concat([df_old_cleaned, df_overlap, df_new_cleaned])

                # ##### Now prepare the final concatenated dataframe for export
                station_name_new = MultiIndex_new.get_level_values("station")[1]
                
                # ! This is where Neil and I made the change to address the issues
                # ! 
                MultiIndex_old = pd.MultiIndex.from_tuples(
                    [(station_name_new, lvl1) for _, lvl1 in MultiIndex_old],
                    names=MultiIndex_new.names,
                )

                MultiIndex_concat = MultiIndex_new.union(MultiIndex_old)

                # drop duplicate rows that were potentially generated in the concatenation process
                df_concat = df_concat.drop_duplicates(subset=["time"])

                # drop 'station' and 'time'columns
                df_concat = df_concat.drop(["station", "time","hour","day","month","year","date"], axis=1)

                print('length of MultiIndex_new')
                print(len(MultiIndex_new))
                print("length of MultiIndex_old")
                print(len(MultiIndex_old))
                print("length of MultiIndex_concat")
                print(len(MultiIndex_concat))

                print("length of df_new")
                print(len(df_new))
                print("length of df_old")
                print(len(df_old))
                print("length of df_concat")
                print(len(df_concat))

                # ! This is where the issue! MultiIndex_concat and df_concat have difference lengths
                df_concat.index = MultiIndex_concat

                # # Convert concatenated dataframe to dataset
                # ds_concat = df_concat.to_xarray()

                # # #### Prepare for export #####

                # # Convert datatype of station coordinate
                # ds_concat.coords["station"] = ds_concat.coords["station"].astype("<U20")

                # # # Include past attributes
                # ds_concat.attrs.update(attrs_new)

                # # Update 'history' attribute
                # timestamp = datetime.datetime.utcnow().strftime("%m-%d-%Y, %H:%M:%S")
                # ds_concat.attrs["history"] = ds_concat.attrs[
                #     "history"
                # ] + " \n maritime_merge.ipynb run on {} UTC".format(timestamp)

                # # Update 'comment' attribute
                # ds_concat.attrs["comment"] = (
                #     "Final v1 data product. This data has been subjected to cleaning, QA/QC, and standardization."
                # )

                # # Add new qaqc_files_merged attribute
                # station_name_old = MultiIndex_old.get_level_values("station")[1]
                # ds_concat.attrs["qaqc_files_merged"] = (
                #     "{}, {} merged. Overlap retained from newer station data.".format(
                #         station_name_old, station_name_new
                #     )
                # )

                # ! this is here the renaming will go

                # !

                # ## Export ###
                # ! a test name is used below
                # ! the final name will be that of the newer dataframe
                # export_url = "s3://wecc-historical-wx/3_qaqc_wx/{}/{}_{}.zarr".format(
                #     network_name, "test_concat", station_name_new
                # )
                # ds_concat.to_zarr(export_url, mode="w")

                # record that the stations were concatenated
                final_concat_list.append(station_1)
                final_concat_list.append(station_2)

            except Exception as e:
                print(
                    "Error concatenating subset {}: {}".format(concat_list, e)
                )
        # if there are more than two stations in the subset, continue
        else:
            continue

    # return final_concat_list # ! this will be the final return statement, below is inlcluded for testing
    # return (
    #     df_new,
    #     df_old,
    #     df_concat,
    #     ds_concat,
    #     final_concat_list,
    # )

    return df_1, df_2, MultiIndex_1, MultiIndex_2, df_concat

### TEST

In [ ]:
network_name = "MARITIME" # "VALLEYWATER", "MARITIME"

In [ ]:
df_1, df_2, MultiIndex_1, MultiIndex_2 = concatenate_station_pairs(network_name)

In [ ]:
# LJAC1 - this should be new
print(df_1['time'].min())
print(df_1["time"].max())

In [ ]:
# LJPC1
print(df_2["time"].min())
print(df_2["time"].max())

In [ ]:
# determine which dataset is older
if df_2["time"].max() > df_1["time"].max():
    # if df_1 has an earlier end tiem than df_2, then d_2 is newer
    # we also grab the name of the newer station in this step, for use later
    df_new = df_2
    MultiIndex_new = MultiIndex_2

    df_old = df_1
    MultiIndex_old = MultiIndex_1

else:
    df_new = df_1
    ds_new = df_1
    MultiIndex_new = MultiIndex_1

    df_old = df_2
    MultiIndex_old = MultiIndex_2

# now set things up to determine if there is temporal overlap between df_new and df_old
df_overlap = df_new[df_new["time"].isin(df_old["time"])]

In [ ]:
# if there is no overlap between the two time series, just concatenate
if len(df_overlap) == 0:
    df_concat = concat([df_old, df_new])

# if not, split into subsets and concatenate
else:
    ##### Split datframes into subsets #####

    # Remove data in time overlap between old and new
    df_old_cleaned = df_old[~df_old["time"].isin(df_overlap["time"])]
    df_new_cleaned = df_new[~df_new["time"].isin(df_overlap["time"])]

    ##### Concatenate subsets #####
    df_concat = concat([df_old_cleaned, df_overlap, df_new_cleaned])

In [ ]:
# ##### Now prepare the final concatenated dataframe for export
station_name_new = MultiIndex_new.get_level_values("station")[1]

MultiIndex_old = pd.MultiIndex.from_tuples(
    [(station_name_new, lvl1) for _, lvl1 in MultiIndex_old],
    names=MultiIndex_new.names,
)

MultiIndex_concat = MultiIndex_new.union(MultiIndex_old)


# MultiIndex_concat = pd.MultiIndex.from_tuples(
#     [(station_name_new, lvl1) for _, lvl1 in MultiIndex_concat],
#     names=MultiIndex_concat.names,
# )

In [ ]:
# drop duplicate rows that were potentially generated in the concatenation process
df_concat = df_concat.drop_duplicates(subset=["time"])

# drop 'station' and 'time'columns
df_concat = df_concat.drop(["station", "time","hour","day","month","year","date"], axis=1)

df_concat.index = MultiIndex_concat

# Convert concatenated dataframe to dataset
ds_concat = df_concat.to_xarray()

In [ ]:
ds_concat

Union is the issue - mismatch in timesteps

In [ ]:
print(df_1['time'].min())
print(df_1["time"].max())

In [ ]:
print(df_2["time"].min())
print(df_2["time"].max())

df_concat should span 2005-01-01 01:30:00 - 2022-08-31 23:54:00

In [ ]:
len(MultiIndex_concat)

In [ ]:
len(df_concat)

In [ ]:
df_concat.columns

In [ ]:
# drop duplicate rows that were potentially generated in the concatenation process
df_concat_drop_dups = df_concat.drop_duplicates(subset=["time"])

In [ ]:
len(df_concat_drop_dups)

In [ ]:
# MultiIndex_concat and df_concat_drop_dups['time']
index_time = list(MultiIndex_1.get_level_values("time"))
#df_time = list(df_concat_drop_dups['time'])

In [ ]:
MultiIndex_1.get_level_values("time")

In [ ]:
df_new['time']

In [ ]:
df_concat['time']

In [ ]:
dups = df_concat[df_concat['time'].duplicated(keep=False)]

In [ ]:
dups

In [ ]:
# drop 'station' and 'time'columns
df_concat = df_concat.drop(
    ["station", "time", "hour", "day", "month", "year", "date"], axis=1
)

df_concat.index = MultiIndex_concat

#### Test option 1

Run concatenate_station_pairs() as is, so the function does not export and instead returns df_concat, df_new, df_old, and df_overlap

In [ ]:
(
    df_new,
    df_old,
    df_concat,
    ds_concat,
    final_concat_list,
) = concatenate_station_pairs(network_name)

In [ ]:
df_concat = df_concat.reset_index(level="time")

#### Test option 2: 

Run concatenate_station_pairs() with the first return statement uncommented and the second commented, and the export section uncommented. So that the function actually exports the concatenated datasets. I've generated all the concatention lists (for VALLEYWATER, MARITIME, and ASOSAWOS) needed to run the function.

In [ ]:
output = concatenate_station_pairs(network_name)

In [ ]:
# import output
# TODO: you'll need to change the url
url_output = "s3://wecc-historical-wx/3_qaqc_wx/{}/test_concat_{}.zarr".format(
    network_name, network_name
)

# TODO: open_zarr will be used for QAQC'd datasets
ds_concat = xr.open_zarr(url_output)

df_concat = ds_concat.to_dataframe()

In [ ]:
network_list = s3_cl.get_object(
    Bucket=bucket,
    Key="3_qaqc_wx/{}/{}_concat_list_{}.csv".format(
        network_name, network_name, network_name
    ),
)
concat_list = pd.read_csv(BytesIO(network_list["Body"].read()))
station_1 = concat_list["ERA-ID"].iloc[0]
station_2 = concat_list["ERA-ID"].iloc[1]

# import this subset of datasets and convert to dataframe
url_1 = "s3://wecc-historical-wx/3_qaqc_wx/{}/{}.zarr".format(network_name, station_1)
url_2 = "s3://wecc-historical-wx/3_qaqc_wx/{}/{}.zarr".format(network_name, station_2)

ds_1 = xr.open_zarr(url_1)
ds_2 = xr.open_zarr(url_2)

df_1 = ds_1.to_dataframe()
df_2 = ds_2.to_dataframe()

In [ ]:
# extract time index for plotting
df_1 = df_1.reset_index(level="time")
df_2 = df_2.reset_index(level="time")


df_concat = df_concat.reset_index(level="time")

In [ ]:
if df_1["time"].max() < df_2["time"].max(): 
    # if df_1 has an earlier end tiem than df_2, then d_2 is newer
    # we also grab the name of the newer station in this step, for use later
    df_new = df_2
    ds_new = ds_2

    df_old = df_1
    ds_old = ds_1
else:
    df_new = df_1
    ds_new = ds_1

    df_old = df_2
    ds_old = ds_2

#### Onward

In [ ]:
ds_concat

In [ ]:
df_concat.head(4)

Check overlap

In [ ]:
# now set things up to determine if there is temporal overlap between df_new and df_old
df_new_overlap = df_new[df_new["time"].isin(df_concat["time"])]
df_concat_overlap = df_concat[df_concat["time"].isin(df_new["time"])]

In [ ]:
df_new_overlap.head(4)

In [ ]:
df_concat_overlap.head(4)

Plot the two original datasets

In [ ]:
vis_var = 'ps'

In [ ]:
# Create a figure with a specific size
plt.figure(figsize=(8, 4))

# Plotting the time series of given dataframe
plt.plot(df_new["time"], df_new[vis_var])

# Plotting the time series of given dataframe
plt.plot(df_old["time"], df_old[vis_var])

# Giving title to the chart using plt.title
plt.title("input dfs")

# rotating the x-axis tick labels at 30degree
# towards right
plt.xticks(rotation=30, ha="right")

# Providing x and y label to the chart
plt.xlabel("time")
plt.ylabel(vis_var)

Plot the output dataset

In [ ]:
# Create a figure with a specific size
plt.figure(figsize=(8, 4))

# Plotting the time series of given dataframe
plt.plot(df_concat["time"], df_concat[vis_var])

# Giving title to the chart using plt.title
plt.title("concatenated df")

# rotating the x-axis tick labels at 30degree
# towards right
plt.xticks(rotation=30, ha="right")

# Providing x and y label to the chart
plt.xlabel("time")
plt.ylabel(vis_var)

## Step 4: Mark stations that have been concatenated